# MàthemaSharp.Quantum — Usage

Complex<T> states, dense operators and finite systems. Prerequisites: Release build, JupyterSharp on .NET 11. Constraint: IEEE + IMinMaxValue for the real component T.

## Contracts
- Normalize accepts a finite, non-null state; it scales the components to avoid premature overflow. Norm can be infinite if the norm exceeds the range of T.
- InnerProduct conjugates the first argument; ExpectationValue keeps the imaginary part and does not automatically normalise the state.
- CheckEigenstate uses a residual relative to the Frobenius norm of the operator; finite non-negative tolerance. It is also valid for non-Hermitian operators.
- DiagonalizeHermitian2 requires a finite 2×2 Hermitian matrix; the phases of the eigenvectors are arbitrary. It is not a general diagonalisation.
- MatrixExp is Taylor with scaling-and-squaring (24 terms by default), not Padé and not an accuracy certifier. TimeEvolution requires finite t and finite positive hbar.


In [ ]:
#load "../shared/MathemaSharp.Numerics.csx"
using System.Numerics;
using System.Runtime.InteropServices;
using MathemaSharp.Quantum;
Console.WriteLine(RuntimeInformation.FrameworkDescription);


.NET 11.0.0-rc.3.26522.148

## Scaled normalisation and eigenstate of X
The ket [MaxValue,MaxValue] is finite, but its norm exceeds the double range. Normalize works on the scaled ket.


In [ ]:
Complex<double>[] huge = [new(double.MaxValue,0),new(double.MaxValue,0)];
if(!double.IsPositiveInfinity(QuantumAlgorithms.Norm<double>(huge))) throw new Exception("Expected overflowing norm");
var unit = QuantumAlgorithms.Normalize<double>(huge);
if(Math.Abs(QuantumAlgorithms.Norm<double>(unit)-1)>1e-12) throw new Exception("Normalization");
var eigen = QuantumAlgorithms.CheckEigenstate<double>(Pauli<double>.X,unit,1e-12);
if(!eigen.IsEigenstate || (eigen.Eigenvalue-Complex<double>.One).GetMagnitude()>1e-12) throw new Exception("Eigenstate");
if(QuantumAlgorithms.CheckEigenstate<double>(Pauli<double>.X,[Complex<double>.One,Complex<double>.Zero],1e-12).IsEigenstate) throw new Exception("False eigenstate");
Console.WriteLine("normalization: OK");
Console.WriteLine("eigenstate_and_rejection: OK");


normalization: OK
eigenstate_and_rejection: OK

## Diagonalisation and density matrix
The eigenvectors of Y are orthogonal; a normalised state produces a density matrix with trace one and purity one.


In [ ]:
var spectral = QuantumAlgorithms.DiagonalizeHermitian2(Pauli<double>.Y);
if(Math.Abs(spectral.LambdaPlus-1)>1e-12 || Math.Abs(spectral.LambdaMinus+1)>1e-12) throw new Exception("Eigenvalues");
if(QuantumAlgorithms.InnerProduct<double>(spectral.VectorPlus,spectral.VectorMinus).GetMagnitude()>1e-12) throw new Exception("Orthogonality");
var rho = QuantumAlgorithms.DensityMatrix<double>(spectral.VectorPlus);
if((rho.Trace()-Complex<double>.One).GetMagnitude()>1e-12 || Math.Abs(QuantumAlgorithms.Purity(rho)-1)>1e-12) throw new Exception("Density");
Console.WriteLine("diagonalization: OK");
Console.WriteLine("density_trace_and_purity: OK");


diagonalization: OK
density_trace_and_purity: OK

## Decimal128 time evolution
For H=X and t=π/2: exp(-iXt)|0⟩ = -i|1⟩. The comparisons are in Decimal128, not converted back to double.


In [ ]:
var one = Decimal128.One;
var two = Decimal128.CreateChecked(2);
var tiny = one;
for(var i=0;i<28;i++) tiny/=Decimal128.CreateChecked(10);
var evolution = QuantumAlgorithms.TimeEvolution(Pauli<Decimal128>.X,Decimal128.Pi/two);
var ket = QuantumAlgorithms.Apply(evolution,QubitStates<Decimal128>.Zero);
if(ket[0].GetMagnitude()>tiny || (ket[1]+Complex<Decimal128>.ImaginaryOne).GetMagnitude()>tiny) throw new Exception("Evolution Decimal128");
if(!evolution.IsUnitary()) throw new Exception("Unitarity");
Console.WriteLine("evolution_decimal128_1e28: OK");


evolution_decimal128_1e28: OK

## Reference and limits
[Public API catalogue](12-api-reference.ipynb) · [Checks on the six formats](11-consolidation.ipynb) · [Inventory](../../ARCHITECTURE.md).

The error below 1e-28 is verified for this specific case, not for every matrix. Unbalanced spectra, non-normal operators and extreme scaling require separate analysis. ComplexMatrix is a struct with a shared backing array: assigning a matrix does not create an independent copy of the data. No infinite quantum simulation or GPU is included.
